## Code related to processing of HumanPPI clustering and search against PDB interfaces

In [1]:
import numpy as np
import pandas as pd
import struct
import matplotlib.pyplot as plt
import seaborn as sns
pd.options.mode.copy_on_write = True

## Human PPI clustering

In [ ]:
# From annotations/add_annot_to_clusters.ipynb
pdb_clusters = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/cluster_analysis/pdb_clusters_annotated.tsv", sep="\t")
# From annotations/create_cluster_summary.ipynb
cluster_summary = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/cluster_analysis/pdb_cluster_summary.tsv", sep="\t")

/var/folders/8n/b4ym5rbn48v7d2lxw_5ph7z40000gp/T/ipykernel_79793/3249868305.py:1: DtypeWarning: Columns (32,34) have mixed types. Specify dtype option on import or set low_memory=False.
  pdb_clusters = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/cluster_analysis/pdb_clusters_annotated.tsv", sep="\t")


In [ ]:
# Results of humanPPI clustering
humanppi_cluster = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/bfmdvspdb/humanppi_only/Result_13subdb_clu_cluster.tsv", sep="\t", header=None)
humanppi_cluster.columns = ["rep","mem"]
# HumanPPI Foldseek database files
humanppi_lookup = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/bfmdvspdb/humanppi_only/humanppi_intdb.lookup", sep="\t", header=None)
humanppi_lookup.columns = ["chain_index","chain_name","complex_id"]
humanppi_id_index = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/bfmdvspdb/humanppi_only/humanppi_intdb_id.index", sep="\t", header=None)
humanppi_id_index.columns = ["chain_index","first","len"]

In [4]:
humanppi_cluster["dimer_id"] = [int(x.split("_")[0][2:]) for x in humanppi_cluster.mem]
humanppi_cluster["cluster_id"] = [int(x.split("_")[0][2:]) for x in humanppi_cluster.rep]
humanppi_lookup["dimer_id"] = humanppi_lookup["complex_id"]

In [5]:
humanppi_cluster["uniprot1"] = [x.split("_")[2] for x in humanppi_cluster.mem]
humanppi_cluster["uniprot2"] = [x.split("_")[5] for x in humanppi_cluster.mem]

In [6]:
humanppi_lookup["cumcount"] = humanppi_lookup.groupby("dimer_id")["chain_name"].cumcount()
humanppi_cluster_melt = pd.melt(humanppi_cluster, id_vars=["dimer_id","cluster_id"], value_vars=["uniprot1","uniprot2"])
humanppi_cluster_melt["cumcount"] = humanppi_cluster_melt.groupby("dimer_id")["variable"].cumcount()
humanppi_ifres = pd.merge(humanppi_lookup, humanppi_cluster_melt[["dimer_id","cluster_id","value","cumcount"]], on=["dimer_id","cumcount"], how="left")
humanppi_ifres = pd.merge(humanppi_ifres, humanppi_id_index, on="chain_index", how="left")
humanppi_ifres

,chain_index,chain_name,complex_id,dimer_id,cumcount,cluster_id,value,first,len
0,74250,Humanppi_A0A075B6H7_S0__A0A0A0MS00_S0_A_B_A,37125,37125,0,37512.0,A0A075B6H7,0,77
1,74251,Humanppi_A0A075B6H7_S0__A0A0A0MS00_S0_A_B_B,37125,37125,1,37512.0,A0A0A0MS00,77,77
2,74252,Humanppi_A0A075B6H7_S0__B9A064_S0_A_B_A,37126,37126,0,37166.0,A0A075B6H7,154,141
3,74253,Humanppi_A0A075B6H7_S0__B9A064_S0_A_B_B,37126,37126,1,37166.0,B9A064,295,57
4,74254,Humanppi_A0A075B6H7_S0__P15814_S0_A_B_A,37127,37127,0,37166.0,A0A075B6H7,352,169
...,...,...,...,...,...,...,...,...,...
41901,116339,Humanppi_Q9Y678_S1__Q9Y689_S0_A_B_B,58169,58169,1,58169.0,Q9Y689,9718461,105
41902,116340,Humanppi_Q9Y6F7_S0__Q9Y6F8_S0_A_B_A,58170,58170,0,58130.0,Q9Y6F7,9718566,265
41903,116341,Humanppi_Q9Y6F7_S0__Q9Y6F8_S0_A_B_B,58170,58170,1,58130.0,Q9Y6F8,9718831,253
41904,116342,Humanppi_S4R3P1_S0__S4R3Y5_S0_A_B_A,58171,58171,0,NaN,NaN,9719084,49


In [7]:
def get_int_list(r, byte_data):
    first = r['first']
    length = r['len']
    curr_window = byte_data[first:(first+length+1)]
    num_integers = len(curr_window) // 4
    integers = list(struct.unpack(f"<{num_integers}I", curr_window[:num_integers*4]))
    return ','.join([str(x) for x in integers])

with open('/Volumes/imb-luckgr/projects/interface_clustering/results/bfmdvspdb/humanppi_only/humanppi_intdb_id', 'rb') as f:
    byte_data = f.read()

humanppi_ifres["if_res"] = humanppi_ifres.apply(lambda x: get_int_list(x, byte_data), axis=1)
humanppi_ifres.rename(columns={"value":"uniprot"}, inplace=True)
humanppi_ifres

,chain_index,chain_name,complex_id,dimer_id,cumcount,cluster_id,uniprot,first,len,if_res
0,74250,Humanppi_A0A075B6H7_S0__A0A0A0MS00_S0_A_B_A,37125,37125,0,37512.0,A0A075B6H7,0,77,"21,57,58,59,61,62,63,64,65,66,67,70,76,77,106,..."
1,74251,Humanppi_A0A075B6H7_S0__A0A0A0MS00_S0_A_B_B,37125,37125,1,37512.0,A0A0A0MS00,77,77,"53,54,55,57,58,59,60,61,62,63,74,102,103,104,1..."
2,74252,Humanppi_A0A075B6H7_S0__B9A064_S0_A_B_A,37126,37126,0,37166.0,A0A075B6H7,154,141,"24,25,26,27,28,29,30,31,32,33,34,35,36,37,39,4..."
3,74253,Humanppi_A0A075B6H7_S0__B9A064_S0_A_B_B,37126,37126,1,37166.0,B9A064,295,57,"101,102,103,104,105,106,107,108,109,110,111,11..."
4,74254,Humanppi_A0A075B6H7_S0__P15814_S0_A_B_A,37127,37127,0,37166.0,A0A075B6H7,352,169,"21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,3..."
...,...,...,...,...,...,...,...,...,...,...
41901,116339,Humanppi_Q9Y678_S1__Q9Y689_S0_A_B_B,58169,58169,1,58169.0,Q9Y689,9718461,105,"16,45,46,47,48,49,50,51,52,53,54,63,65,66,67,6..."
41902,116340,Humanppi_Q9Y6F7_S0__Q9Y6F8_S0_A_B_A,58170,58170,0,58130.0,Q9Y6F7,9718566,265,"6,23,37,38,39,40,41,42,43,44,46,148,149,150,15..."
41903,116341,Humanppi_Q9Y6F7_S0__Q9Y6F8_S0_A_B_B,58170,58170,1,58130.0,Q9Y6F8,9718831,253,"6,21,39,40,41,42,43,44,45,46,131,132,133,134,1..."
41904,116342,Humanppi_S4R3P1_S0__S4R3Y5_S0_A_B_A,58171,58171,0,NaN,NaN,9719084,49,"4,5,6,7,8,9,10,11,12,14,15,16"


In [ ]:
# Average ipLDDT for each interface
humanppi_plddt = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/bfmdvspdb/humanppi_only/name_avgintplddt_length", sep="\t", header=None)
humanppi_plddt.columns = ["complex_name","mean_iplddt","if_len"]
humanppi_plddt["dimer_id"] = [int(x.split("_")[0].split("DI")[1]) for x in humanppi_plddt.complex_name]
humanppi_ifres["mean_iplddt"] = humanppi_ifres["dimer_id"].map(dict(zip(humanppi_plddt.dimer_id, humanppi_plddt.mean_iplddt)))

humanppi_ifres.loc[:, "ipLDDTabove70"] = 0
humanppi_ifres.loc[humanppi_ifres["mean_iplddt"] >= 70, "ipLDDTabove70"] = 1

clustermean_iplddt = humanppi_ifres[humanppi_ifres["mean_iplddt"] > 0].groupby("cluster_id")["mean_iplddt"].mean().to_frame()
clustermean_iplddt.columns = ["mean_meaniplddt"]
clustermean_iplddt.reset_index(inplace=True)
humanppi_cluster["mean_meaniplddt"] = humanppi_cluster["cluster_id"].map(dict(zip(clustermean_iplddt.cluster_id, clustermean_iplddt.mean_meaniplddt)))

clustermax_iplddt = humanppi_ifres[humanppi_ifres["mean_iplddt"] > 0].groupby("cluster_id")["mean_iplddt"].max().to_frame()
clustermax_iplddt.columns = ["max_meaniplddt"]
clustermax_iplddt.reset_index(inplace=True)
humanppi_cluster["max_meaniplddt"] = humanppi_cluster["cluster_id"].map(dict(zip(clustermax_iplddt.cluster_id, clustermax_iplddt.max_meaniplddt)))

print(humanppi_ifres["ipLDDTabove70"].sum() / 2)
print(humanppi_cluster[humanppi_cluster["max_meaniplddt"] >= 70].shape[0])

11438.0
13852


In [ ]:
# Use this file for get_humanppi_coiledcoils.py and get_humanppi_disorder.py
humanppi_ifres.to_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/bfmdvspdb/humanppi_only/humanppi_ifres.tsv", index=None, sep="\t")

In [ ]:
# From get_humanppi_coiledcoils.py
humanppi_coils = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/bfmdvspdb/humanppi_uniprot_coiledcoils.tsv", sep="\t")
# From get_humanppi_disorder.py
humanppi_disorder = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/bfmdvspdb/humanppi_disorder_fractions.tsv", sep="\t")

In [ ]:
humanppi_coils.dropna(subset="ft_coiled", axis=0, inplace=True)
print(humanppi_coils.shape[0])
humanppi_coils["range"] = [x.split("COILED")[-1].split(";")[0] for x in humanppi_coils["ft_coiled"]]

# Keep coiled-coil annotations only if domain range overlaps with at least one interface residue
def check_coil_range(r):
    coil_in_range = np.nan
    if isinstance(r["if_res"], str):
        try:
            full_range = [int(x) for x in str(r['coil_range']).split('..')]
        except:
            return 0
        if len(full_range) < 2:
            return 0
        coil_in_range = 0
        for ifres in [int(x) for x in r['if_res'].split(',') if ((x != 'nan') & (x != ''))]:
            if ifres in full_range:
                coil_in_range = 1
                break
    return coil_in_range

def return_coil_sum(x):
    return sum(x["coil_in_range"])

humanppi_ifres["coil_range"] = humanppi_ifres["uniprot"].map(dict(zip(humanppi_coils.uniprot_id, humanppi_coils.range)))
humanppi_ifres_coilsonly = humanppi_ifres.dropna(subset="coil_range", axis=0)
#res_map.drop("range", axis=1, inplace=True)
humanppi_ifres_coilsonly["coil_in_range"] = humanppi_ifres_coilsonly.apply(lambda x: check_coil_range(x), axis=1)

coil_list = humanppi_ifres_coilsonly.groupby("complex_id").apply(lambda x: return_coil_sum(x), include_groups=False)
coil_list = coil_list.to_frame()
coil_list.columns = ["coil_status"]
coil_list[coil_list["coil_status"] == 2]

humanppi_cluster_plusannot = pd.merge(humanppi_cluster, coil_list, left_on="dimer_id", right_index=True, how="left")
humanppi_cluster_plusannot.loc[:, "coil_status_label"] = "Not Coiled-coil"
humanppi_cluster_plusannot.loc[humanppi_cluster_plusannot["coil_status"] >= 1, "coil_status_label"] = "Coiled-coil"

humanppi_cluster_plusannot[humanppi_cluster_plusannot["coil_status_label"] == "Coiled-coil"]

1910


,rep,mem,dimer_id,cluster_id,uniprot1,uniprot2,mean_meaniplddt,max_meaniplddt,coil_status,coil_status_label
20,DI54738_Humanppi_Q7Z7H5_S0__Q9BVK6_S0,DI54738_Humanppi_Q7Z7H5_S0__Q9BVK6_S0,54738,54738,Q7Z7H5,Q9BVK6,60.61240,60.6124,1.0,Coiled-coil
22,DI54739_Humanppi_Q7Z7H5_S0__Q9Y3A6_S0,DI48681_Humanppi_P49755_S0__Q13445_S0,48681,54739,P49755,Q13445,77.44520,84.4259,1.0,Coiled-coil
23,DI54739_Humanppi_Q7Z7H5_S0__Q9Y3A6_S0,DI48682_Humanppi_P49755_S0__Q15363_S0,48682,54739,P49755,Q15363,77.44520,84.4259,1.0,Coiled-coil
31,DI54739_Humanppi_Q7Z7H5_S0__Q9Y3A6_S0,DI54741_Humanppi_Q7Z7H5_S0__Q9Y3Q3_S0,54741,54739,Q7Z7H5,Q9Y3Q3,77.44520,84.4259,1.0,Coiled-coil
35,DI54739_Humanppi_Q7Z7H5_S0__Q9Y3A6_S0,DI57287_Humanppi_Q9BVK6_S0__Q9Y3Q3_S0,57287,54739,Q9BVK6,Q9Y3Q3,77.44520,84.4259,1.0,Coiled-coil
...,...,...,...,...,...,...,...,...,...,...
20245,DI54648_Humanppi_Q7Z4T9_S0__Q9UFE4_S2,DI54648_Humanppi_Q7Z4T9_S0__Q9UFE4_S2,54648,54648,Q7Z4T9,Q9UFE4,72.29360,72.2936,1.0,Coiled-coil
20273,DI54684_Humanppi_Q7Z6B7_S2__Q9HCE6_S1,DI54684_Humanppi_Q7Z6B7_S2__Q9HCE6_S1,54684,54684,Q7Z6B7,Q9HCE6,NaN,NaN,1.0,Coiled-coil
20274,DI54685_Humanppi_Q7Z6B7_S2__Q9HCE6_S2,DI54685_Humanppi_Q7Z6B7_S2__Q9HCE6_S2,54685,54685,Q7Z6B7,Q9HCE6,61.42340,61.4234,1.0,Coiled-coil
20276,DI54687_Humanppi_Q7Z6B7_S2__Q9UIW2_S3,DI54687_Humanppi_Q7Z6B7_S2__Q9UIW2_S3,54687,54687,Q7Z6B7,Q9UIW2,NaN,NaN,1.0,Coiled-coil


In [11]:
print(humanppi_disorder.shape[0])
humanppi_disorder.drop_duplicates(subset=['complex_id','chain_id'], inplace=True, keep=False)
print(humanppi_disorder.shape[0])
humanppi_disorder.dropna(subset=['disfrac'], inplace=True, axis=0)
print(humanppi_disorder.shape[0])
humanppi_disorder['idx'] = humanppi_disorder.groupby('complex_id').cumcount()
print(humanppi_disorder[humanppi_disorder['idx'] > 1])
humanppi_disorder = humanppi_disorder.pivot(index='complex_id', columns='idx')['disfrac']
humanppi_disorder = humanppi_disorder.sort_index(axis=1, level=1)
humanppi_disorder.columns = ['disfrac_0','disfrac_1']
humanppi_disorder = humanppi_disorder.reset_index()
humanppi_disorder.dropna(subset=['disfrac_0','disfrac_1'], inplace=True, axis=0)
print(humanppi_disorder.shape[0])
humanppi_disorder['maxdisfrac'] = np.nanmax(humanppi_disorder[['disfrac_0','disfrac_1']].values, axis=1)
humanppi_disorder['mindisfrac'] = np.nanmin(humanppi_disorder[['disfrac_0','disfrac_1']].values, axis=1)

41844
41844
41733
Empty DataFrame
Columns: [chain_id, complex_id, uniprot_id, disfrac, idx]
Index: []
20781


In [ ]:
# Create orderedness categories for interfaces
humanppi_cluster_plusannot["maxdisfrac"] = humanppi_cluster_plusannot["dimer_id"].map(dict(zip(humanppi_disorder.complex_id, humanppi_disorder.maxdisfrac)))
humanppi_cluster_plusannot["mindisfrac"] = humanppi_cluster_plusannot["dimer_id"].map(dict(zip(humanppi_disorder.complex_id, humanppi_disorder.mindisfrac)))

humanppi_cluster_plusannot.loc[((humanppi_cluster_plusannot['maxdisfrac'] > 0.5) & (humanppi_cluster_plusannot['mindisfrac'] > 0.5)), "if_type"] = "Disorder-disorder"
humanppi_cluster_plusannot.loc[((humanppi_cluster_plusannot['maxdisfrac'] > 0.5) & (humanppi_cluster_plusannot['mindisfrac'] <= 0.5)), "if_type"] = "Disorder-order"
humanppi_cluster_plusannot.loc[((humanppi_cluster_plusannot['maxdisfrac'] <= 0.5) & (humanppi_cluster_plusannot['mindisfrac'] <= 0.5)), "if_type"] = "Order-order"
humanppi_cluster_plusannot['if_type'] = humanppi_cluster_plusannot['if_type'].fillna("Unannotated")
num_order_if = humanppi_cluster_plusannot[humanppi_cluster_plusannot['if_type'] == "Order-order"].shape[0]
num_disord_if = humanppi_cluster_plusannot[humanppi_cluster_plusannot['if_type'] == "Disorder-order"].shape[0]
num_dis_if = humanppi_cluster_plusannot[humanppi_cluster_plusannot['if_type'] == "Disorder-disorder"].shape[0]
num_unchar_if = humanppi_cluster_plusannot[humanppi_cluster_plusannot['if_type'] == "Unannotated"].shape[0]
print("Number of order-order interactions: ", num_order_if, f"({round(num_order_if/20953*100, 1)}%)")
print("Number of disorder-order interactions: ", num_disord_if, f"({round(num_disord_if/20953*100, 1)}%)")
print("Number of disorder-disorder interactions: ", num_dis_if, f"({round(num_dis_if/20953*100, 1)}%)")
print("Number of unannotated interactions: ", num_unchar_if, f"({round(num_unchar_if/20953*100, 1)}%)")

Number of order-order interactions:  16897 (80.6%)
Number of disorder-order interactions:  2979 (14.2%)
Number of disorder-disorder interactions:  254 (1.2%)
Number of unannotated interactions:  169 (0.8%)


In [ ]:
# HumanPPI cluster size
humanppi_cluster_size = humanppi_cluster_plusannot.groupby("cluster_id")["mem"].size().to_frame()
humanppi_cluster_size.rename(columns={"mem":"cluster_size"}, inplace=True)
humanppi_cluster_plussize = pd.merge(humanppi_cluster_plusannot, humanppi_cluster_size, left_on="cluster_id", right_index=True, how="left")

def get_if_len(r):
    ifres = [int(x) for x in str(r["if_res"]).split(",") if x != 'nan']
    iflen = len(ifres)
    return iflen

humanppi_ifres["if_len"] = humanppi_ifres.apply(lambda x: get_if_len(x), axis=1)
humanppi_ifsize = humanppi_ifres.groupby("complex_id")["if_len"].max().to_frame()
humanppi_cluster_plussize["max_if_len"] = humanppi_cluster_plussize["dimer_id"].map(dict(zip(humanppi_ifsize.index, humanppi_ifsize.if_len)))
humanppi_cluster_plussize

,rep,mem,dimer_id,cluster_id,uniprot1,uniprot2,mean_meaniplddt,max_meaniplddt,coil_status,coil_status_label,maxdisfrac,mindisfrac,if_type,cluster_size,max_if_len
0,DI54711_Humanppi_Q7Z6L1_S2__Q9Y4F3_S2,DI54711_Humanppi_Q7Z6L1_S2__Q9Y4F3_S2,54711,54711,Q7Z6L1,Q9Y4F3,65.3859,65.3859,NaN,Not Coiled-coil,0.412500,0.172043,Order-order,1,93
1,DI54712_Humanppi_Q7Z6M4_S0__Q96CB9_S0,DI54712_Humanppi_Q7Z6M4_S0__Q96CB9_S0,54712,54712,Q7Z6M4,Q96CB9,NaN,NaN,NaN,Not Coiled-coil,0.173913,0.000000,Order-order,1,25
2,DI54716_Humanppi_Q7Z6V5_S0__Q96EY9_S0,DI54716_Humanppi_Q7Z6V5_S0__Q96EY9_S0,54716,54716,Q7Z6V5,Q96EY9,87.2261,87.2261,NaN,Not Coiled-coil,0.280702,0.000000,Order-order,1,57
3,DI54717_Humanppi_Q7Z6W1_S0__Q8IZ16_S0,DI54717_Humanppi_Q7Z6W1_S0__Q8IZ16_S0,54717,54717,Q7Z6W1,Q8IZ16,46.9439,46.9439,0.0,Not Coiled-coil,0.000000,0.000000,Order-order,1,33
4,DI54719_Humanppi_Q7Z769_S0__Q99942_S0,DI54719_Humanppi_Q7Z769_S0__Q99942_S0,54719,54719,Q7Z769,Q99942,NaN,NaN,NaN,Not Coiled-coil,0.130435,0.000000,Order-order,1,80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20294,DI54705_Humanppi_Q7Z6J9_S0__Q9BSV6_S0,DI54705_Humanppi_Q7Z6J9_S0__Q9BSV6_S0,54705,54705,Q7Z6J9,Q9BSV6,NaN,NaN,NaN,Not Coiled-coil,0.462366,0.217391,Order-order,1,93
20295,DI54706_Humanppi_Q7Z6K4_S0__Q8NB46_S2,DI54706_Humanppi_Q7Z6K4_S0__Q8NB46_S2,54706,54706,Q7Z6K4,Q8NB46,94.4275,94.4275,NaN,Not Coiled-coil,0.000000,0.000000,Order-order,1,28
20296,DI54707_Humanppi_Q7Z6L0_S0__Q9BZC5_S0,DI54707_Humanppi_Q7Z6L0_S0__Q9BZC5_S0,54707,54707,Q7Z6L0,Q9BZC5,NaN,NaN,NaN,Not Coiled-coil,0.000000,0.000000,Order-order,1,30
20297,DI54708_Humanppi_Q7Z6L1_S1__Q9Y4F3_S1,DI54708_Humanppi_Q7Z6L1_S1__Q9Y4F3_S1,54708,54708,Q7Z6L1,Q9Y4F3,67.8787,67.8787,NaN,Not Coiled-coil,0.068966,0.036364,Order-order,1,58


In [ ]:
# Results from humanPPI alignment against PDB interface cluster representatives (includes cases where only one interface chain matches)
with open("/Volumes/imb-luckgr/projects/interface_clustering/results/bfmdvspdb/humanppi_only/humanppinames_qtm0.4_OR_ttm0.4", "r") as f:
    humanppi_singlechain_hits = [x.strip("\n") for x in f.readlines()]

# Results from humanPPI alignment against PDB interface cluster representatives (includes only cases where both interface chains match)
with open("/Volumes/imb-luckgr/projects/interface_clustering/results/bfmdvspdb/humanppi_only/wholehumanppi_pdb_searchres_report_qtm0.4_OR_ttm0.4_two_chain_matched.tsv", "r") as f:
    humanppi_doublechain_hits = [x.split("\t")[0] for x in f.readlines()]

humanppi_cluster_plussize.loc[:,"pdb_singlechain_hit"] = 0
humanppi_cluster_plussize.loc[humanppi_cluster_plussize["mem"].isin(humanppi_singlechain_hits), "pdb_singlechain_hit"] = 1

humanppi_cluster_plussize.loc[:,"pdb_doublechain_hit"] = 0
humanppi_cluster_plussize.loc[humanppi_cluster_plussize["mem"].isin(humanppi_doublechain_hits), "pdb_doublechain_hit"] = 1

In [ ]:
# For use in figure4_S4.ipynb
humanppi_cluster_plussize.to_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/bfmdvspdb/humanppi_cluster_plusannot.tsv", sep="\t", index=None)

## Filtering clusters for manual inspection

In [16]:
humanppi_cluster_hits = humanppi_cluster_plussize.groupby("cluster_id")[["pdb_singlechain_hit","pdb_doublechain_hit"]].sum()
humanppi_cluster_hits.columns = ["pdb_singlechain_hit","pdb_doublechain_hit"]
humanppi_cluster_hits["cluster_size"] = humanppi_cluster_hits.index.map(dict(zip(humanppi_cluster_plussize.cluster_id, humanppi_cluster_plussize.cluster_size)))

humanppi_cluster_hits["mean_meaniplddt"] = humanppi_cluster_hits.index.map(dict(zip(humanppi_cluster_plussize["cluster_id"], humanppi_cluster_plussize.mean_meaniplddt)))
humanppi_cluster_hits["max_meaniplddt"] = humanppi_cluster_hits.index.map(dict(zip(humanppi_cluster_plussize["cluster_id"], humanppi_cluster_plussize.max_meaniplddt)))

In [ ]:
# Inspect clusters with no match at all (not even single-chain matches)
# Focus only on clusters with highly-confident predicted interfaces (ipLDDT > 70)
cluster_ids_to_examine = list(set(humanppi_cluster_hits[humanppi_cluster_hits["pdb_singlechain_hit"] == 0].index))
humanppi_cluster_nohits = humanppi_cluster_plussize[((humanppi_cluster_plussize["cluster_id"].isin(cluster_ids_to_examine)) & (humanppi_cluster_plussize["max_meaniplddt"] >= 70))]
print(humanppi_cluster_nohits["coil_status_label"].value_counts())
print(humanppi_cluster_nohits["if_type"].value_counts())
print(humanppi_cluster_nohits["max_if_len"].describe())
print(humanppi_cluster_nohits["cluster_size"].describe())

coil_status_label
Not Coiled-coil    38
Name: count, dtype: int64
if_type
Order-order          29
Disorder-order        8
Disorder-disorder     1
Name: count, dtype: int64
count    38.000000
mean     29.684211
std      15.436190
min      13.000000
25%      17.250000
50%      24.500000
75%      40.750000
max      76.000000
Name: max_if_len, dtype: float64
count    38.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
Name: cluster_size, dtype: float64


In [44]:
humanppi_cluster_nohits[humanppi_cluster_nohits["if_type"] == "Disorder-disorder"]

,rep,mem,dimer_id,cluster_id,uniprot1,uniprot2,mean_meaniplddt,max_meaniplddt,coil_status,coil_status_label,maxdisfrac,mindisfrac,if_type,cluster_size,max_if_len,pdb_singlechain_hit,pdb_doublechain_hit
16217,DI48635_Humanppi_P49639_S0__Q9UQ80_S0,DI48635_Humanppi_P49639_S0__Q9UQ80_S0,48635,48635,P49639,Q9UQ80,78.3469,78.3469,NaN,Not Coiled-coil,0.75,0.6875,Disorder-disorder,1,32,0,0


In [45]:
humanppi_cluster_nohits[humanppi_cluster_nohits["if_type"] == "Disorder-order"].sort_values(by="max_if_len", ascending=False)

,rep,mem,dimer_id,cluster_id,uniprot1,uniprot2,mean_meaniplddt,max_meaniplddt,coil_status,coil_status_label,maxdisfrac,mindisfrac,if_type,cluster_size,max_if_len,pdb_singlechain_hit,pdb_doublechain_hit
11609,DI51242_Humanppi_Q12788_S1__Q86WX3_S0,DI51242_Humanppi_Q12788_S1__Q86WX3_S0,51242,51242,Q12788,Q86WX3,72.3145,72.3145,NaN,Not Coiled-coil,0.608696,0.058824,Disorder-order,1,51,0,0
8020,DI49460_Humanppi_P55735_S0__Q9NQW1_S2,DI49460_Humanppi_P55735_S0__Q9NQW1_S2,49460,49460,P55735,Q9NQW1,75.0265,75.0265,NaN,Not Coiled-coil,0.555556,0.000000,Disorder-order,1,50,0,0
17319,DI45088_Humanppi_P12270_S3__Q8NHY2_S0,DI45088_Humanppi_P12270_S3__Q8NHY2_S0,45088,45088,P12270,Q8NHY2,74.9629,74.9629,0.0,Not Coiled-coil,1.000000,0.000000,Disorder-order,1,34,0,0
14937,DI47347_Humanppi_P30405_S0__P55854_S0,DI47347_Humanppi_P30405_S0__P55854_S0,47347,47347,P30405,P55854,71.4578,71.4578,NaN,Not Coiled-coil,1.000000,0.040000,Disorder-order,1,25,0,0
3133,DI43232_Humanppi_P02458_S2__Q6ZMI3_S0,DI43232_Humanppi_P02458_S2__Q6ZMI3_S0,43232,43232,P02458,Q6ZMI3,79.7887,79.7887,NaN,Not Coiled-coil,0.833333,0.000000,Disorder-order,1,24,0,0
1852,DI41666_Humanppi_O75717_S1__Q92771_S2,DI41666_Humanppi_O75717_S1__Q92771_S2,41666,41666,O75717,Q92771,75.4962,75.4962,NaN,Not Coiled-coil,0.666667,0.111111,Disorder-order,1,18,0,0
8249,DI49737_Humanppi_P60866_S0__Q86VM9_S0,DI49737_Humanppi_P60866_S0__Q86VM9_S0,49737,49737,P60866,Q86VM9,74.9417,74.9417,0.0,Not Coiled-coil,1.000000,0.000000,Disorder-order,1,17,0,0
10240,DI40425_Humanppi_O43747_S2__P63010_S1,DI40425_Humanppi_O43747_S2__P63010_S1,40425,40425,O43747,P63010,72.8956,72.8956,NaN,Not Coiled-coil,1.000000,0.000000,Disorder-order,1,13,0,0


In [46]:
humanppi_cluster_nohits[humanppi_cluster_nohits["if_type"] == "Order-order"].sort_values(by="max_if_len", ascending=False)

,rep,mem,dimer_id,cluster_id,uniprot1,uniprot2,mean_meaniplddt,max_meaniplddt,coil_status,coil_status_label,maxdisfrac,mindisfrac,if_type,cluster_size,max_if_len,pdb_singlechain_hit,pdb_doublechain_hit
12819,DI52516_Humanppi_Q15678_S2__Q92626_S1,DI52516_Humanppi_Q15678_S2__Q92626_S1,52516,52516,Q15678,Q92626,70.1535,70.1535,NaN,Not Coiled-coil,0.307692,0.092105,Order-order,1,76,0,0
2240,DI42200_Humanppi_O94985_S2__Q63HQ2_S2,DI42200_Humanppi_O94985_S2__Q63HQ2_S2,42200,42200,O94985,Q63HQ2,83.8263,83.8263,NaN,Not Coiled-coil,0.390625,0.017544,Order-order,1,64,0,0
13707,DI52914_Humanppi_Q2VWP7_S2__Q8NFZ4_S0,DI52914_Humanppi_Q2VWP7_S2__Q8NFZ4_S0,52914,52914,Q2VWP7,Q8NFZ4,72.7593,72.7593,NaN,Not Coiled-coil,0.200000,0.120000,Order-order,1,50,0,0
16681,DI44315_Humanppi_P08648_S2__P78324_S0,DI44315_Humanppi_P08648_S2__P78324_S0,44315,44315,P08648,P78324,77.9210,77.9210,NaN,Not Coiled-coil,0.272727,0.062500,Order-order,1,48,0,0
1739,DI41541_Humanppi_O75473_S1__P17181_S0,DI41541_Humanppi_O75473_S1__P17181_S0,41541,41541,O75473,P17181,79.6526,79.6526,NaN,Not Coiled-coil,0.000000,0.000000,Order-order,1,42,0,0
19542,DI53809_Humanppi_Q6EMK4_S0__Q9Y219_S1,DI53809_Humanppi_Q6EMK4_S0__Q9Y219_S1,53809,53809,Q6EMK4,Q9Y219,77.9167,77.9167,NaN,Not Coiled-coil,0.133333,0.095238,Order-order,1,42,0,0
15626,DI47944_Humanppi_P40818_S2__P60484_S0,DI47944_Humanppi_P40818_S2__P60484_S0,47944,47944,P40818,P60484,85.0834,85.0834,NaN,Not Coiled-coil,0.121951,0.103448,Order-order,1,41,0,0
12267,DI51862_Humanppi_Q14112_S1__Q9Y6N6_S1,DI51862_Humanppi_Q14112_S1__Q9Y6N6_S1,51862,51862,Q14112,Q9Y6N6,72.3960,72.3960,0.0,Not Coiled-coil,0.048780,0.000000,Order-order,1,41,0,0
16008,DI48349_Humanppi_P46781_S0__Q9BY44_S0,DI48349_Humanppi_P46781_S0__Q9BY44_S0,48349,48349,P46781,Q9BY44,85.8366,85.8366,0.0,Not Coiled-coil,0.407407,0.000000,Order-order,1,40,0,0
4893,DI38617_Humanppi_O00206_S1__Q99467_S0,DI38617_Humanppi_O00206_S1__Q99467_S0,38617,38617,O00206,Q99467,78.3845,78.3845,NaN,Not Coiled-coil,0.000000,0.000000,Order-order,1,39,0,0


In [ ]:
# Now find clusters with some single-chain matches but no double-chain matches
# Focus on larger clusters and on clusters with highly-confident predicted interfaces (cluster average ipLDDT > 70)
partialmatch_cluster_ids_to_examine = list(set(humanppi_cluster_hits[((humanppi_cluster_hits["pdb_doublechain_hit"] == 0) & (humanppi_cluster_hits["pdb_singlechain_hit"] > 0))].index))
humanppi_cluster_partialhits_nosing = humanppi_cluster_plussize[((humanppi_cluster_plussize["cluster_id"].isin(partialmatch_cluster_ids_to_examine)) & (humanppi_cluster_plussize["max_meaniplddt"] >= 70) & (humanppi_cluster_plussize["cluster_size"] > 1))]
print(humanppi_cluster_partialhits_nosing.drop_duplicates(subset=["rep"])["rep"][0:20])

146         DI54850_Humanppi_Q86V59_S0__Q96BY2_S0
862         DI55726_Humanppi_Q8NEW7_S0__Q8TDI8_S0
1847        DI41661_Humanppi_O75716_S0__Q10567_S1
2225        DI42185_Humanppi_O94973_S2__P61966_S0
2596        DI42700_Humanppi_O95881_S0__Q9Y3D2_S0
2745        DI42853_Humanppi_P00533_S1__Q9HCU5_S0
2807        DI42914_Humanppi_P01042_S0__P20151_S0
4097    DI37619_Humanppi_A0A1B0GX56_S0__P06331_S0
4129    DI37665_Humanppi_A0A1W2PQC6_S0__Q9BQ65_S0
4661        DI38226_Humanppi_A8MPP1_S2__O75717_S1
4703        DI38309_Humanppi_A8MXD5_S0__Q9NZA1_S0
6100        DI56652_Humanppi_Q96FG2_S0__Q9H0F7_S0
6197        DI56807_Humanppi_Q96LK0_S0__Q9UBK7_S0
6511        DI57165_Humanppi_Q9BQK8_S0__Q9Y6K0_S0
7435        DI48846_Humanppi_P51116_S0__Q6P2E9_S1
7626        DI49017_Humanppi_P51805_S2__Q13214_S0
7861        DI49269_Humanppi_P54198_S1__Q9NPG3_S0
7927        DI49350_Humanppi_P54764_S1__Q15375_S2
8086        DI49538_Humanppi_P56706_S0__Q8J025_S0
8244        DI49733_Humanppi_P60842_S0__Q13347_S0


In [48]:
humanppi_cluster_plussize[humanppi_cluster_plussize["mem"] == "DI55236_Humanppi_Q8IZ81_S0__Q9NVJ2_S0"]

,rep,mem,dimer_id,cluster_id,uniprot1,uniprot2,mean_meaniplddt,max_meaniplddt,coil_status,coil_status_label,maxdisfrac,mindisfrac,if_type,cluster_size,max_if_len,pdb_singlechain_hit,pdb_doublechain_hit
6103,DI56652_Humanppi_Q96FG2_S0__Q9H0F7_S0,DI55236_Humanppi_Q8IZ81_S0__Q9NVJ2_S0,55236,56652,Q8IZ81,Q9NVJ2,90.53175,93.3677,NaN,Not Coiled-coil,0.0,0.0,Order-order,4,31,1,0


In [ ]:
# Export list of Uniprot IDs for functional enrichment analysis in enrichGO_humanppi_nohit.R
clusters_to_enrich = list(set(humanppi_cluster_hits[humanppi_cluster_hits["status"] != "Full PDB Match"].index))
ifs_to_enrich = humanppi_cluster_plussize[humanppi_cluster_plussize["cluster_id"].isin(clusters_to_enrich)]
uniprots_to_enrich = list(set(ifs_to_enrich["uniprot1"]) | set(ifs_to_enrich["uniprot2"]))
with open("/Users/stromjoe/Documents/humanppi_nohit_uniprots.txt", "w+") as f:
    f.write(",".join(uniprots_to_enrich))